<a href="https://colab.research.google.com/github/Anshu-code-202/ai-knowledge-graph-infosys_springborad-batch13-training/blob/main/Milestone_3_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

base_path = "/content/drive/MyDrive/AI_KG_Project/data/processed/"

nodes = pd.read_csv(base_path + "graph_nodes.csv")
edges = pd.read_csv(base_path + "graph_edges.csv")

print(" Graph loaded — Milestone-3 ready")

 Graph loaded — Milestone-3 ready


In [3]:
import pandas as pd

# Set your base folder path
base_path = "/content/drive/MyDrive/AI_KG_Project/data/processed/"

# Load Milestone-1 outputs
customers_clean = pd.read_csv(base_path + "clean_customers.csv")
orders_clean = pd.read_csv(base_path + "clean_orders.csv")
products_clean = pd.read_csv(base_path + "clean_products.csv")
sellers_clean = pd.read_csv(base_path + "clean_sellers.csv")

customer_order_rel = pd.read_csv(base_path + "rel_customer_order.csv")
order_product_rel = pd.read_csv(base_path + "rel_order_product.csv")
seller_product_rel = pd.read_csv(base_path + "rel_seller_product.csv")

print(" Milestone-1 data loaded")

 Milestone-1 data loaded


In [4]:
nodes.to_csv(base_path + "graph_nodes.csv", index=False)
edges.to_csv(base_path + "graph_edges.csv", index=False)

print(" Milestone-2 Completed — Graph Saved")

 Milestone-2 Completed — Graph Saved


In [5]:
!pip install langchain sentence-transformers

In [6]:
# Import Libraries
get_ipython().system('pip install --upgrade langchain-text-splitters')
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# Create Text Documents from Data

# Convert your structured data into text descriptions.

documents = []

for _, row in orders_clean.iterrows():

    text = f"""
    Order ID: {row['order_id']}
    Customer ID: {row['customer_id']}
    Order Status: {row['order_status']}
    Purchase Timestamp: {row['order_purchase_timestamp']}
    """

    documents.append(text)

print("Total documents created:", len(documents))

In [ ]:
# Apply Text Chunking

# We split large documents into smaller chunks.
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = text_splitter.create_documents(documents)

print("Total chunks created:\n", len(chunks))

In [ ]:
print(chunks[0])

Embedding Generation

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
!pip install neo4j
from neo4j import GraphDatabase

# Replace with your Neo4j URI and credentials
uri = "neo4j+s://2c12eff8.databases.neo4j.io"
username = "2c12eff8"
password = "92qjvLKrudBOMz1rxY8O5LP8IU86zBueX6eR-tqpzDU"

driver = GraphDatabase.driver(uri, auth=(username, password))

orders_list = orders_clean.sample(n=100, random_state=42).to_dict("records") # Drastically reduce sample to avoid node limit issues

query = """
UNWIND $rows AS row
MERGE (o:Order {order_id: row.order_id})
SET o.order_status = row.order_status,
    o.order_purchase_timestamp = row.order_purchase_timestamp
"""

batch_size = 50 # Adjusted batch size

with driver.session() as session:
    for i in range(0, len(orders_list), batch_size):
        batch = orders_list[i:i+50]
        session.run(query, rows=batch)

driver.close()

print("Orders loaded successfully!")

In [ ]:
get_ipython().system('pip install neo4j')
print("Neo4j Python driver ensured.")

from neo4j import GraphDatabase

# IMPORTANT: Please verify these connection details directly from your Neo4j Aura console.
# An incorrect URI, username, or password will cause connection failures.
# Also, ensure your Neo4j Aura instance is running and accessible.
uri = "neo4j+s://2c12eff8.databases.neo4j.io"
username = "2c12eff8"
password = "92qjvLKrudBOMz1rxY8O5LP8IU86zBueX6eR-tqpzDU"

try:
    driver = GraphDatabase.driver(uri, auth=(username, password))
    driver.verify_connectivity()
    print("Neo4j driver initialized and connectivity verified.")
except Exception as e:
    print(f"Failed to connect to Neo4j: {e}")
    print("Please check your Neo4j Aura URI, username, password, and ensure the instance is running.")
    # Re-raise the exception or exit if connection is critical for subsequent steps
    raise

with driver.session() as session:
    # 1. Clear all existing data
    session.run("MATCH (n) DETACH DELETE n")
    print("Neo4j database cleared.")

    # 2. Create unique constraint for Customer nodes
    session.run("CREATE CONSTRAINT customer_id_constraint IF NOT EXISTS FOR (c:Customer) REQUIRE c.customer_id IS UNIQUE")
    print("Customer ID constraint created.")

    # 3. Create unique constraint for Order nodes
    session.run("CREATE CONSTRAINT order_id_constraint IF NOT EXISTS FOR (o:Order) REQUIRE o.order_id IS UNIQUE")
    print("Order ID constraint created.")

    # 4. Create unique constraint for Product nodes
    session.run("CREATE CONSTRAINT product_id_constraint IF NOT EXISTS FOR (p:Product) REQUIRE p.product_id IS UNIQUE")
    print("Product ID constraint created.")

    # 5. Create unique constraint for Seller nodes
    session.run("CREATE CONSTRAINT seller_id_constraint IF NOT EXISTS FOR (s:Seller) REQUIRE s.seller_id IS UNIQUE")
    print("Seller ID constraint created.")

print("All constraints defined and database prepared for data loading.")


In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss
import numpy as np

In [ ]:
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import random

# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Sample chunks to avoid memory/time issues during embedding generation
# Using a fixed random seed for reproducibility
random.seed(42)
sample_size = 1000 # You can adjust this number
sampled_chunks = random.sample(chunks, min(len(chunks), sample_size))

# Generate embeddings for each sampled chunk with a progress bar
embeddings = model.encode([chunk.page_content for chunk in tqdm(sampled_chunks, desc="Generating Embeddings")])

# Print information
print("Total sampled chunks:", len(sampled_chunks))
print("Shape of embeddings:", embeddings.shape)

# Print sample embedding
print("Sample embedding (first 5 values):")
print(embeddings[0][:5])

In [ ]:
def rag_pipeline(query, model, index, chunks, k=3):

  #step 1:query->embedding:
  query_embedding = model.encode([query]).astype("float32")

  # Step 2: Search in FAISS
  distances,indices=index.search(query_embedding,k)

  # Step 3: Retrieve chunks
  reterived_texts=[chunks[i].page_content for i in indices[0]]

  # Step 4: Build Context
  context="\n".join(reterived_texts)

  # Step 5: Generate answer using LLM
  answer=generate_answer(context,query)

  return answer

In [ ]:
# Prepare Embeddings:Your embeddings are already generated like this:

# Using sampled_chunks for efficiency, as generating embeddings for all 99441 chunks can take a very long time.
embeddings = model.encode([chunk.page_content for chunk in sampled_chunks])

# Convert them to NumPy float32 (FAISS requires this).
embeddings = np.array(embeddings).astype("float32")
print(f"Generated embeddings for {len(sampled_chunks)} sampled chunks. Shape: {embeddings.shape}")

In [ ]:
# Step 1: Create sample chunks
sample_chunks = chunks[:1000]

# Step 2: Generate embeddings
embeddings = model.encode([chunk.page_content for chunk in sample_chunks])

# Step 3: Create FAISS index
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

Save embeddings once:

In [ ]:
import numpy as np
np.save("embeddings.npy", embeddings)


Create FAISS Index
First find the dimension of embeddings.

In [ ]:
dimension = embeddings.shape[1]

# Create FAISS index:
index = faiss.IndexFlatL2(dimension)
# IndexFlatL2 → uses Euclidean distance for similarity search# Works well for small to medium datasets

# Add embeddings to the FAISS index
index.add(embeddings)

print("Embeddings successfully added to FAISS index.")
print("Total vectors stored in FAISS:", index.ntotal)

In [ ]:
index.add(embeddings)

Perform Similarity Search (Test)

In [ ]:
query = "late delivery issue"

query_embedding = model.encode([query]).astype("float32")

k = 3  # top results

distances, indices = index.search(query_embedding, k)

Retrieve Similar Text Chunks

In [ ]:
for i in indices[0]:
    print(chunks[i].page_content)
    print("----")

## Build Context from Retrieved Chunks


In [ ]:
retrived_texts = [chunks[i].page_content for i in indices[0]]
context = "\n".join(retrived_texts)
print(context)

Create Prompt for LLM
Now you pass context + user query to an LLM.

Multi-Query System (No single query bcz it must be scalable)

In [ ]:
query = "which orders had delivery issues?"
prompt=f"""
Answer the question based on context below.

Context:{context}

Question:{query}"""

In [ ]:
def generate_answer(context,query):
  prompt=f"""
  You are a helpful assistant.

  Answer the question based only on the context.
  If answer is not present,say "Not found in data".

  Context:
  {context}

  Question:
  {query}
  """


  # Avoids wrong answers
# Makes system more reliable

Generate Answer (LLM Step)
Now the LLM generates the final answer.

You can use:

OpenAI API

HuggingFace models

Local LLM

In [ ]:
print(prompt)

In [ ]:
!pip install transfomers

LLM Answer Generation code
Proper Prompt + LLM Integration

we already have:

chunks

model (SentenceTransformer)

faiss index

Now add LLM step :

In [ ]:
from transformers import pipeline

#load a simple llm(text generation)
llm=pipeline("question-answering",model="google/flan-t5-base")
def generate_answer(context,query):
  # Temporarily return the full result object to inspect 'score' and 'answer'
  result = llm(question=query, context=context)
  return result

Clean Pipeline Function (VERY IMPORTANT)
Now combine everything into one function 🔹 Final Clean RAG Pipeline

In [ ]:
def rag_pipeline(query,model,index,chunks,k=3):

  #step 1:query->embedding:
  query_embedding = model.encode([query]).astype("float32")

  # Step 2: Search in FAISS
  distances,indices=index.search(query_embedding,k)

  # Step 3: Retrieve chunks
  reterived_texts=[chunks[i].page_content for i in indices[0]]

  # Step 4: Build Context
  context="\n".join(reterived_texts)

  # Step 5: Generate answer using LLM
  answer=generate_answer(context,query)

  return answer

In [ ]:
query="Which orders had delivery issues?"
answer=rag_pipeline(query,model,index,chunks, k=5)
print("Full LLM Result:\n",answer)

In [ ]:
print(len(chunks))

In [ ]:
print(type(chunks[0]))

In [ ]:
# Now here system handles unlimited queries
import time

while True:
  query = input("Ask your question(type ' exit' to stop):")

  if query.lower() == "exit":
    break

  start_time = time.time()
  answer = rag_pipeline(query,model,index,chunks)
  end_time = time.time()
  print(f"\nAnswer generated in {end_time - start_time:.2f} seconds.")
  print("\n Answer:\n",answer)